# Part 4.1 — Embeddings & Similarity
**Topics covered:** Cricket, Cooking, Cybersecurity  
**Model:** `all-MiniLM-L6-v2` from `sentence-transformers`

In [ ]:
# Install required libraries
!pip install sentence-transformers seaborn matplotlib -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully.")

## Step 1: Define 10 sentences across 3 topics

In [ ]:
sentences = [
    # Cricket (4 sentences) — indices 0-3
    "The batsman hit a massive six over the boundary.",
    "India won the test match by an innings and 50 runs.",
    "The spinner bowled a perfect googly to dismiss the opener.",
    "Rain interrupted play and the match was reduced to 20 overs.",

    # Cooking (3 sentences) — indices 4-6
    "Saute the onions in olive oil until they turn golden brown.",
    "Marinating the chicken overnight makes it juicy and flavorful.",
    "Always preheat the oven before baking a cake.",

    # Cybersecurity (3 sentences) — indices 7-9
    "A phishing attack tricks users into revealing their passwords.",
    "Firewalls monitor incoming and outgoing network traffic to block threats.",
    "End-to-end encryption ensures that only the sender and receiver can read the message."
]

labels = [
    "Cricket-1", "Cricket-2", "Cricket-3", "Cricket-4",
    "Cooking-1", "Cooking-2", "Cooking-3",
    "CyberSec-1", "CyberSec-2", "CyberSec-3"
]

print(f"Total sentences: {len(sentences)}")
for i, s in enumerate(sentences):
    print(f"  [{labels[i]}] {s}")

## Step 2: Generate Embeddings using all-MiniLM-L6-v2

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(sentences)

print(f"Embedding shape: {embeddings.shape}")
print(f"Each sentence is represented as a {embeddings.shape[1]}-dimensional vector.")

## Step 3: Compute 10×10 Cosine Similarity Matrix & Heatmap

In [ ]:
cos_sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix (10x10):")
print(np.round(cos_sim_matrix, 3))

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    cos_sim_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='white'
)
plt.title("Cosine Similarity Matrix — Cricket, Cooking, Cybersecurity", fontsize=13, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("similarity_heatmap.png", dpi=150)
plt.show()
print("Heatmap saved as similarity_heatmap.png")

## Step 4: Query — Find Top 2 Most Similar Sentences

In [ ]:
query = "The bowler took three wickets in one over"
query_embedding = model.encode([query])

# Compute similarity between query and all 10 sentences
query_similarities = cosine_similarity(query_embedding, embeddings)[0]

# Get top 2 indices
top2_indices = np.argsort(query_similarities)[::-1][:2]

print(f"Query: \"{query}\"")
print("-" * 60)
print("Top 2 most similar sentences:\n")
for rank, idx in enumerate(top2_indices, 1):
    print(f"  Rank {rank}: [{labels[idx]}]")
    print(f"  Sentence : {sentences[idx]}")
    print(f"  Similarity Score: {query_similarities[idx]:.4f}")
    print()

## Observations

- **High similarity within topics**: Cricket sentences cluster together, as do Cooking and Cybersecurity sentences — visible as warm blocks on the diagonal of the heatmap.
- **Low cross-topic similarity**: Sentences from different topics score near 0, showing that semantic embeddings capture topic meaning effectively.
- **Query result**: The query about a bowler taking wickets correctly matches Cricket sentences with the highest scores, validating the model's semantic understanding.